# Q5. 手推一步 Policy Gradient 更新

给定：模型对动作 $a$ 的概率 $\pi = 0.3$，reward $R = +2$。

Policy Gradient loss 为：$L = -R \cdot \log \pi$

1. 手算 $L$ 的数值
2. 手算 $\frac{\partial L}{\partial \pi}$（对概率求偏导）
3. 如果学习率 $\text{lr} = 0.1$，新的 $\pi$ 应该变大还是变小？变成多少？

$$L = -R \cdot \log(\pi) = ? \qquad \frac{\partial L}{\partial \pi} = ? \qquad \pi_{\text{new}} = \pi - \text{lr} \cdot \frac{\partial L}{\partial \pi} = ?$$

**验收标准**：数值正确，且能说清楚「正 reward 让概率上升」的方向。

> **简化说明**：本题将 $\pi$ 视为直接可更新的参数，目的是看清梯度方向。实际训练中更新的是网络权重 $\theta$，$\pi$ 由 softmax 得出。

## 手算推导

**Step 1：计算 loss**

$$L = -R \cdot \log \pi = -2 \times \log(0.3) = -2 \times (-1.204) = \boxed{2.408}$$

**Step 2：求梯度**

$$\frac{\partial L}{\partial \pi} = -\frac{R}{\pi} = -\frac{2}{0.3} = \boxed{-6.667}$$

梯度为负，说明沿梯度方向（减小 $\pi$）会让 loss 下降；反过来，**梯度下降会增大 $\pi$**。

**Step 3：更新**

$$\pi_{\text{new}} = \pi - \text{lr} \cdot \frac{\partial L}{\partial \pi} = 0.3 - 0.1 \times (-6.667) = 0.3 + 0.667 = \boxed{0.967}$$

**直觉**：reward 是正的 $\Rightarrow$ 这个动作是好的 $\Rightarrow$ 应该提高它的概率。loss $= -R \cdot \log \pi$ 的梯度恰好实现了这一点：梯度为负，更新后 $\pi$ 变大。

In [1]:
import torch

pi = torch.tensor(0.3, requires_grad=True)
R  = torch.tensor(2.0)
lr = 0.1

# Forward
L = -R * torch.log(pi)
print(f"L = {L.item():.4f}  (expected: 2.4079)")

# Backward
L.backward()
print(f"dL/dpi = {pi.grad.item():.4f}  (expected: -6.6667)")

# Update
with torch.no_grad():
    pi_new = pi - lr * pi.grad
print(f"pi_new = {pi_new.item():.4f}  (expected: 0.9667)")
print(f"Direction correct (pi increased for positive reward): {pi_new.item() > pi.item()}")

L = 2.4079  (expected: 2.4079)
dL/dpi = -6.6667  (expected: -6.6667)
pi_new = 0.9667  (expected: 0.9667)
Direction correct (pi increased for positive reward): True


## 用 PyTorch 计算 log 概率

词表大小为 5，模型输出 `logits = [1.0, 2.0, 0.5, -1.0, 0.3]`。

用 PyTorch 完成：

1. 对 logits 做 softmax，得到概率分布
2. 假设真实选择的 token 是 `index=1`，取出它的概率
3. 计算 log 概率（用 `torch.log`）
4. 用 `torch.nn.functional.cross_entropy` 验证（`cross_entropy = -log π`）

**验收标准**：log prob 与 `-cross_entropy` 数值一致。

## 为什么 cross_entropy = -log π？

### cross_entropy 是什么，p 和 q 分别是谁

交叉熵的定义：

$$H(p, q) = -\sum_i p_i \log q_i$$

- $p$ 是**真实分布**，也就是「正确答案」——在分类任务里，我们知道正确类别是哪个，所以 $p$ 是一个 one-hot 向量。比如词表大小为 5、正确 token 是 index=1，那么 $p = [0, 1, 0, 0, 0]$
- $q$ 是**模型预测的分布**，也就是 softmax 之后的概率向量。在本题里 $q = \text{softmax}(\text{logits})$，比如 $q = [0.12, 0.47, 0.08, 0.01, 0.06, ...]$

### 为什么一展开就只剩一项

把 $p = [0, 1, 0, 0, 0]$ 代入求和：

$$H(p, q) = -\sum_i p_i \log q_i = -(0 \cdot \log q_0 + 1 \cdot \log q_1 + 0 \cdot \log q_2 + 0 \cdot \log q_3 + 0 \cdot \log q_4)$$

$$= -\log q_1 = -\log \pi_1$$

所以 **cross_entropy 就是对正确类别的预测概率取负 log**，其他类别的项因为 $p_i = 0$ 全部消掉了。

### 本题中为什么相等

`logits = [1.0, 2.0, 0.5, -1.0, 0.3]`，正确 token 是 `index=1`，代入上面的结论：

$$\text{cross\_entropy} = -\log \underbrace{\text{softmax(logits)}[1]}_{\pi_1}$$

而我们手算的 `log_prob` 正是 $\log \pi_1$，所以：

$$\text{log\_prob} = -\text{cross\_entropy}$$

### `F.cross_entropy` 在 PyTorch 里做了什么

它把三步合成一个数值稳定的操作：

$$\text{cross\_entropy}(\text{logits}, y) = -\text{logits}[y] + \log \sum_j e^{\text{logits}[j]}$$

等价于先 softmax 再取 log 再加负号，但数值更稳定（避免 softmax 的上溢/下溢）。

### 和 Policy Gradient 的联系

Policy Gradient 的 loss 是 $L = -R \cdot \log \pi$，当 $R=1$ 时就退化成 cross_entropy。**语言模型的训练（最大化下一个 token 的 log 概率）本质上就是 reward 恒为 1 的 Policy Gradient。**

In [ ]:
import torch
import torch.nn.functional as F

logits = torch.tensor([1.0, 2.0, 0.5, -1.0, 0.3])
target = torch.tensor([1])

# Step 1: softmax → probability distribution
probs = torch.softmax(logits, dim=0)
print("probs:", probs)

# Step 2: pick probability of the chosen token (index=1)
pi = probs[1]
print(f"pi (index=1) = {pi.item():.6f}")

# Step 3: log probability
log_prob = torch.log(pi)
print(f"log_prob = {log_prob.item():.6f}")

# Step 4: verify with cross_entropy  (cross_entropy = -log π)
ce = F.cross_entropy(logits.unsqueeze(0), target)
print(f"cross_entropy = {ce.item():.6f}")
print(f"Match: {torch.allclose(log_prob, -ce)}")